In [ ]:
# %%pip install -q ultralytics opencv-python-headless scipy tqdm

import os, re, json, math, random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from scipy.optimize import linear_sum_assignment
from pathlib import Path
from tqdm import tqdm

IMAGES_DIR   = "/content/drive/MyDrive/pig-selected_frame_attribute_(4)/images_frame_attribute_(4)"
WEIGHTS      = "/content/weights.pt"
MASK_PATH    = "/content/drive/MyDrive/pig_data_unzipped/mask.png"
CLASS_NAME   = "Pig"
BURST_SIZE   = 6

# YOLO
CONF_THRESH  = 0.25
NMS_IOU      = 0.70

# ROI filter
ROI_MODE         = "center"   # "center" hoặc "cover"
ROI_MIN_COVER    = 0.10       # khi ROI_MODE="cover": tối thiểu % bbox area nằm trong ROI
ROI_DILATE_PX    = 8          # nới ROI để không cắt mất box khi chạy sát biên

# Matching 6 frame
IOU_STRICT   = 0.80   # vòng ghép nghiêm
IOU_RELAXED  = 0.60   # vòng ghép bổ sung
COST_THR     = 0.60   # ngưỡng cost cho Hungarian
ALPHA, BETA, GAMMA = 0.6, 0.3, 0.1  # cost = α*(1−IoU) + β*dist + γ*|log area_ratio|

# Hysteresis & motion tube
TAU_ENTER    = 0.50   # ngưỡng điểm detect để “vào” track
TAU_STAY     = 0.25   # ngưỡng thấp hơn để “ở lại” tube
SEARCH_MARGIN= 0.20   # nới rộng 20% quanh box dự báo
KEEPALIVE_MAX= 2      # giữ sống tối đa 2 frame liên tiếp khi miss

# Ổn định đầu ra
PERSIST_MIN  = 5      # tối thiểu 5/6 frame
MAX_PER_BURST= 8      # tối đa 8 ID/burst

# Preview
PREVIEW_BURSTS = 10
SEED           = 42


In [ ]:
# ====== Helpers ======
IMG_EXT = (".jpg",".jpeg",".png",".bmp",".tif",".tiff")
NAME_RE = re.compile(
    r'^(?P<prefix>burst_color_[^_]+_\d+)_f(?P<f>[0-9Oo]+)_k(?P<k>\d+)\.(jpg|jpeg|png|bmp|tif|tiff)$',
    re.IGNORECASE
)

def read_mask(mask_path):
    m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    assert m is not None, f"Cannot read mask: {mask_path}"
    _, m = cv2.threshold(m, 127, 255, cv2.THRESH_BINARY)
    # nới ROI một chút để không cắt box ở biên
    if ROI_DILATE_PX > 0:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ROI_DILATE_PX, ROI_DILATE_PX))
        m = cv2.dilate(m, kernel, iterations=1)
    return m

def scan_bursts(images_dir, burst_size=6):
    files = [str(p) for p in Path(images_dir).iterdir()
             if p.is_file() and p.suffix.lower() in IMG_EXT]
    groups = {}
    for fp in files:
        name = os.path.basename(fp)
        m = NAME_RE.match(name)
        if not m:
            continue
        prefix = m.group("prefix")
        k = m.group("k")
        try:
            k_int = int(k)
        except:
            continue
        groups.setdefault(prefix, []).append((k_int, fp))
    bursts = []
    for prefix, lst in groups.items():
        lst.sort(key=lambda x: x[0])
        for i in range(0, len(lst), burst_size):
            chunk = lst[i:i+burst_size]
            if len(chunk) == burst_size:
                bursts.append([fp for _, fp in chunk])
    return bursts

def detect_batch(model, paths, conf=0.25, iou=0.7):
    out = {}
    res = model(paths, conf=conf, iou=iou, verbose=False)
    for im_path, r in zip(paths, res):
        preds = []
        if r.boxes is not None and len(r.boxes) > 0:
            for b in r.boxes:
                preds.append({
                    "xyxy": b.xyxy[0].cpu().numpy().tolist(),
                    "score": float(b.conf[0].cpu().item()),
                    "cls": int(b.cls[0].cpu().item())
                })
        out[im_path] = preds
    return out

def bbox_iou_xyxy(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    area_a = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    area_b = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    return inter / (area_a + area_b - inter + 1e-6)

def bbox_center_area_xyxy(b):
    x1,y1,x2,y2 = b
    cx = 0.5*(x1+x2); cy = 0.5*(y1+y2)
    area = max(1.0, (x2-x1)*(y2-y1))
    return (cx, cy, area)

def norm_center_dist(c1, c2, W, H):
    return ((c1[0]-c2[0])**2 + (c1[1]-c2[1])**2) ** 0.5 / ( (W*W + H*H) ** 0.5 + 1e-6 )

def xyxy_to_xywh(b):
    x1,y1,x2,y2 = b
    return [float(x1), float(y1), float(x2-x1), float(y2-y1)]

def center_in_roi(mask, box):
    h, w = mask.shape[:2]
    x1,y1,x2,y2 = box
    cx = int((x1+x2)/2.0); cy = int((y1+y2)/2.0)
    cx = max(0, min(w-1, cx)); cy = max(0, min(h-1, cy))
    return mask[cy, cx] == 255

def cover_ratio(mask, box):
    h, w = mask.shape[:2]
    x1,y1,x2,y2 = map(int, [box[0], box[1], box[2], box[3]])
    x1 = max(0, min(w-1, x1)); x2 = max(0, min(w-1, x2))
    y1 = max(0, min(h-1, y1)); y2 = max(0, min(h-1, y2))
    if x2 <= x1 or y2 <= y1: return 0.0
    roi = mask[y1:y2, x1:x2]
    if roi.size == 0: return 0.0
    return np.count_nonzero(roi==255) / float(roi.size)

def roi_keep(mask, box):
    if ROI_MODE == "center":
        return center_in_roi(mask, box)
    return cover_ratio(mask, box) >= ROI_MIN_COVER

# ====== Optical flow (LK) để ước lượng dx,dy trong box ======
def lk_displacement(prev_img, curr_img, prev_box):
    x1,y1,x2,y2 = map(int, prev_box)
    x1 = max(0, min(prev_img.shape[1]-1, x1))
    y1 = max(0, min(prev_img.shape[0]-1, y1))
    x2 = max(0, min(prev_img.shape[1]-1, x2))
    y2 = max(0, min(prev_img.shape[0]-1, y2))
    if x2 <= x1 or y2 <= y1:
        return 0.0, 0.0

    prev_gray = cv2.cvtColor(prev_img, cv2.COLOR_BGR2GRAY)
    curr_gray = cv2.cvtColor(curr_img, cv2.COLOR_BGR2GRAY)
    roi = prev_gray[y1:y2, x1:x2]
    if roi.size < 25:
        return 0.0, 0.0

    pts = cv2.goodFeaturesToTrack(roi, maxCorners=50, qualityLevel=0.01, minDistance=4)
    if pts is None:
        return 0.0, 0.0
    pts = pts.reshape(-1,1,2)
    # chuyển sang coord tuyệt đối
    pts[:,:,0] += x1
    pts[:,:,1] += y1
    pts2, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, pts, None,
                                             winSize=(15,15), maxLevel=2,
                                             criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))
    if pts2 is None or st is None:
        return 0.0, 0.0
    good = st.reshape(-1) == 1
    if not np.any(good):
        return 0.0, 0.0
    d = (pts2[good,:,:] - pts[good,:,:]).reshape(-1,2)
    dx = float(np.median(d[:,0])) if d.shape[0] else 0.0
    dy = float(np.median(d[:,1])) if d.shape[0] else 0.0
    return dx, dy

def make_search_box(box, vxvy, W, H, margin=SEARCH_MARGIN):
    x1,y1,x2,y2 = box
    dx, dy = vxvy
    x1 += dx; x2 += dx; y1 += dy; y2 += dy
    cx = (x1+x2)/2; cy=(y1+y2)/2; w=(x2-x1); h=(y2-y1)
    x1 = cx - (1+margin)*w/2; x2 = cx + (1+margin)*w/2
    y1 = cy - (1+margin)*h/2; y2 = cy + (1+margin)*h/2
    return [max(0,x1), max(0,y1), min(W-1,x2), min(H-1,y2)]

def accept_detection(score, inside_tube):
    if inside_tube and score >= TAU_STAY: return True
    return score >= TAU_ENTER

# Strict/Relaxed + Hysteresis + LK fallback + Persistence + Gap-fill
def build_stable_tracks_6_motion(frame_dets, frame_imgs, frame_sizes):
    """
    frame_dets[t] : list of dict {"box": [x1,y1,x2,y2], "score": float}
    frame_imgs[t] : BGR image np.ndarray
    frame_sizes[t]: (W,H)
    return: final_tracks (list of list[6 x xyxy]), scores (list)
    """
    T = 6
    # Khởi tạo từ frame 0
    tracks = []
    for d in frame_dets[0]:
        tracks.append({
            "boxes": {0: d["box"]},
            "last_box": d["box"],
            "miss": 0,
            "vxvy": (0.0, 0.0),   # vận tốc ước lượng
            "obs": 1              # số frame quan sát thực
        })

    def try_assign_for_frame(tracks, t, iou_gate):
        W,H = frame_sizes[t]
        dets = frame_dets[t]
        used = set()

        # Xây cost so với last_box
        for tr in tracks:
            if (t-1) not in tr["boxes"]:
                continue
            prev_box = tr["boxes"][t-1]
            # search-box theo vận tốc
            sb = make_search_box(prev_box, tr["vxvy"], W, H, SEARCH_MARGIN)

            best_j, best_cost, best_iou = None, 1e9, 0.0
            for j, d in enumerate(dets):
                if j in used:
                    continue
                box = d["box"]; score = d["score"]
                iou = bbox_iou_xyxy(prev_box, box)
                cx1,cy1,_ = bbox_center_area_xyxy(prev_box)
                cx2,cy2,_ = bbox_center_area_xyxy(box)
                dn  = norm_center_dist((cx1,cy1,1.0), (cx2,cy2,1.0), W, H)
                a1 = max(1.0, (prev_box[2]-prev_box[0])*(prev_box[3]-prev_box[1]))
                a2 = max(1.0, (box[2]-box[0])*(box[3]-box[1]))
                ar = (a2+1e-6)/(a1+1e-6)
                inside = (box[0] >= sb[0] and box[1] >= sb[1] and box[2] <= sb[2] and box[3] <= sb[3])

                # chấp nhận theo hysteresis
                if iou < iou_gate:
                    # nếu IoU thấp, vẫn có thể nhận nếu ở trong tube và score ≥ τ_stay
                    if not accept_detection(score, inside):
                        continue
                else:
                    if not accept_detection(score, inside):
                        continue

                cost = ALPHA*(1.0 - iou) + BETA*dn + GAMMA*abs(math.log(ar))
                if cost < best_cost:
                    best_cost, best_j, best_iou = cost, j, iou

            # Gán nếu tốt
            if best_j is not None and best_cost <= COST_THR:
                j = best_j; d = dets[j]; box = d["box"]
                used.add(j)
                # cập nhật vận tốc
                px,py,_ = bbox_center_area_xyxy(prev_box)
                cx,cy,_ = bbox_center_area_xyxy(box)
                tr["vxvy"] = (cx-px, cy-py)
                tr["boxes"][t] = box
                tr["last_box"] = box
                tr["miss"] = 0
                tr["obs"] += 1

        # Tạo track mới cho det chưa dùng (chỉ ở vòng relaxed)
        return used

    # Pass 1: STRICT
    for t in range(1, T):
        used = try_assign_for_frame(tracks, t, IOU_STRICT)
        # Với các track vẫn miss ở t, thử LK fallback/keepalive
        dets = frame_dets[t]; W,H = frame_sizes[t]
        for tr in tracks:
            if t in tr["boxes"]:
                continue
            if (t-1) not in tr["boxes"]:
                continue
            prev_box = tr["boxes"][t-1]
            dx, dy = lk_displacement(frame_imgs[t-1], frame_imgs[t], prev_box)
            # dự báo box
            pb = [prev_box[0]+dx, prev_box[1]+dy, prev_box[2]+dx, prev_box[3]+dy]
            # thử tìm detection gần pb để “stay”
            best_j, best_cost = None, 1e9
            for j, d in enumerate(dets):
                if j in used:
                    continue
                box = d["box"]; score = d["score"]
                inside = (box[0] >= pb[0]-1 and box[1] >= pb[1]-1 and box[2] <= pb[2]+1 and box[3] <= pb[3]+1)
                if not accept_detection(score, inside):
                    continue
                # cost so với pb
                iou = bbox_iou_xyxy(pb, box)
                cx1,cy1,_ = bbox_center_area_xyxy(pb)
                cx2,cy2,_ = bbox_center_area_xyxy(box)
                dn  = norm_center_dist((cx1,cy1,1.0), (cx2,cy2,1.0), W, H)
                a1 = max(1.0, (pb[2]-pb[0])*(pb[3]-pb[1]))
                a2 = max(1.0, (box[2]-box[0])*(box[3]-box[1]))
                ar = (a2+1e-6)/(a1+1e-6)
                cost = ALPHA*(1.0 - iou) + BETA*dn + GAMMA*abs(math.log(ar))
                if cost < best_cost:
                    best_cost, best_j = cost, j
            if best_j is not None and best_cost <= COST_THR:
                d = dets[best_j]; box = d["box"]
                used.add(best_j)
                px,py,_ = bbox_center_area_xyxy(prev_box)
                cx,cy,_ = bbox_center_area_xyxy(box)
                tr["vxvy"] = (cx-px, cy-py)
                tr["boxes"][t] = box
                tr["last_box"] = box
                tr["miss"] = 0
                tr["obs"] += 1
            else:
                # keepalive bằng pb
                tr["boxes"][t] = pb
                tr["last_box"] = pb
                tr["miss"] += 1

    # Pass 2: RELAXED (bổ sung/điều chỉnh những chỗ còn trống)
    for t in range(1, T):
        used = try_assign_for_frame(tracks, t, IOU_RELAXED)
        # không cần keepalive lần nữa (đã có ở pass strict)

        # Sinh track mới từ det chưa dùng (relaxed)
        dets = frame_dets[t]
        for j, d in enumerate(dets):
            if j in used:
                continue
            if d["score"] >= TAU_ENTER:
                tracks.append({
                    "boxes": {t: d["box"]},
                    "last_box": d["box"],
                    "miss": 0,
                    "vxvy": (0.0, 0.0),
                    "obs": 1
                })

    # Lọc theo persistence
    keep = []
    for tr in tracks:
        if len(tr["boxes"]) >= PERSIST_MIN:
            keep.append(tr)
    if not keep:
        return [], []

    # Gap-fill đủ 6 frame (nội suy/ngoại suy)
    def fill_track(tr):
        seq = [None]*T
        for t,b in tr["boxes"].items():
            seq[t] = b
        last = None
        for t in range(T):
            if seq[t] is not None:
                last = t; continue
            nxt = None
            for u in range(t+1, T):
                if seq[u] is not None:
                    nxt = u; break
            if last is not None and nxt is not None:
                # nội suy theo cx,cy,w,h
                def to_c(b):
                    x1,y1,x2,y2=b; return [(x1+x2)/2.0,(y1+y2)/2.0,max(1.0,x2-x1),max(1.0,y2-y1)]
                def from_c(c):
                    cx,cy,w,h=c; return [cx-w/2, cy-h/2, cx+w/2, cy+h/2]
                c_prev = to_c(seq[last]); c_next = to_c(seq[nxt])
                r = (t-last)/float(nxt-last+1e-6)
                c_t = [ c_prev[i]*(1-r) + c_next[i]*r for i in range(4) ]
                seq[t] = from_c(c_t)
            elif last is not None:
                seq[t] = seq[last]
            elif nxt is not None:
                seq[t] = seq[nxt]
            else:
                return None
        return seq

    filled, scores = [], []
    for tr in keep:
        seq = fill_track(tr)
        if seq is None:
            continue
        ious = [bbox_iou_xyxy(seq[t], seq[t+1]) for t in range(T-1)]
        stable = tr["obs"] + float(np.mean(ious))
        filled.append(seq); scores.append(stable)

    # Cắt ≤ MAX_PER_BURST và khử trùng lặp
    if len(filled) > MAX_PER_BURST:
        order = np.argsort(scores)[::-1][:MAX_PER_BURST]
        filled = [filled[i] for i in order]
        scores = [scores[i] for i in order]

    n = len(filled); keep_mask = [True]*n
    for i in range(n):
        if not keep_mask[i]: continue
        for j in range(i+1, n):
            if not keep_mask[j]: continue
            overlap_cnt = sum(1 for t in range(T) if bbox_iou_xyxy(filled[i][t], filled[j][t]) > 0.7)
            if overlap_cnt >= 3:
                if scores[i] >= scores[j]:
                    keep_mask[j] = False
                else:
                    keep_mask[i] = False

    final_tracks = [filled[i] for i in range(n) if keep_mask[i]]
    final_scores = [scores[i] for i in range(n) if keep_mask[i]]
    return final_tracks, final_scores


In [ ]:
random.seed(SEED)
model = YOLO(WEIGHTS)
base_mask = read_mask(MASK_PATH)

all_bursts = scan_bursts(IMAGES_DIR, BURST_SIZE)
assert len(all_bursts) > 0, "Không tìm thấy burst 6 frame theo pattern tên."
sample_bursts = random.sample(all_bursts, k=min(PREVIEW_BURSTS, len(all_bursts)))

GRID_R, GRID_C = len(sample_bursts), BURST_SIZE
plt.figure(figsize=(2.8*GRID_C, 2.8*GRID_R))

for r, burst in enumerate(tqdm(sample_bursts, desc="Preview bursts")):
    det_map = detect_batch(model, burst, conf=CONF_THRESH, iou=NMS_IOU)

    frame_dets, frame_imgs, frame_sizes = [], [], []
    for im_path in burst:
        img = cv2.imread(im_path); assert img is not None, f"Cannot read {im_path}"
        H, W = img.shape[:2]
        m = cv2.resize(base_mask, (W, H), interpolation=cv2.INTER_NEAREST)
        raw = det_map[im_path]
        dets = []
        for p in raw:
            box = p["xyxy"]
            if roi_keep(m, box):
                dets.append({"box": box, "score": p["score"]})
        frame_dets.append(dets)
        frame_imgs.append(img)
        frame_sizes.append((W, H))

    tracks, _ = build_stable_tracks_6_motion(frame_dets, frame_imgs, frame_sizes)

    for c, im_path in enumerate(burst):
        img = frame_imgs[c].copy()
        for local_id, track in enumerate(tracks, start=1):
            x1,y1,x2,y2 = map(int, track[c])
            color = (37*local_id % 255, 97*local_id % 255, 173*local_id % 255)
            cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
            cv2.putText(img, f"ID {local_id}", (x1, max(0,y1-5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax = plt.subplot(GRID_R, GRID_C, int(r*GRID_C + c + 1))
        ax.imshow(img); ax.set_axis_off()
        ax.set_title(os.path.basename(im_path), fontsize=7)

plt.tight_layout()
plt.show()


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
OUT_JSON = IMAGES_DIR + "/annotate_id.coco.json"

model = YOLO(WEIGHTS)
base_mask = read_mask(MASK_PATH)
all_bursts = scan_bursts(IMAGES_DIR, BURST_SIZE)

images, annotations = [], []
categories = [{"id": 1, "name": CLASS_NAME}]  # phải là "Pig" đúng như CVAT của bạn
ann_id = 1
img_id_map = {}
img_id_counter = 1

for burst in tqdm(all_bursts, desc="Exporting COCO"):
    det_map = detect_batch(model, burst, conf=CONF_THRESH, iou=NMS_IOU)

    frame_dets, frame_imgs, frame_sizes = [], [], []
    for im_path in burst:
        img = cv2.imread(im_path); assert img is not None, f"Cannot read {im_path}"
        H, W = img.shape[:2]
        m = cv2.resize(base_mask, (W, H), interpolation=cv2.INTER_NEAREST)
        raw = det_map[im_path]
        dets = []
        for p in raw:
            box = p["xyxy"]
            if roi_keep(m, box):
                dets.append({"box": box, "score": p["score"]})
        frame_dets.append(dets); frame_imgs.append(img); frame_sizes.append((W, H))

    # Ổn định 6 frame (motion-aware)
    tracks, _ = build_stable_tracks_6_motion(frame_dets, frame_imgs, frame_sizes)

    # Ghi COCO
    for t, im_path in enumerate(burst):
        if im_path not in img_id_map:
            H, W = frame_imgs[t].shape[:2]
            img_id_map[im_path] = img_id_counter
            images.append({
                "id": img_id_counter,
                "file_name": os.path.relpath(im_path, IMAGES_DIR),
                "width": W, "height": H
            })
            img_id_counter += 1
        image_id = img_id_map[im_path]

        for local_id, track in enumerate(tracks, start=1):
            lid = int(max(1, min(8, local_id)))  # 1..8
            x1,y1,x2,y2 = track[t]
            bbox = [float(x1), float(y1), float(x2-x1), float(y2-y1)]

            # Attributes KHỚP schema CVAT của bạn
            attr = {
                "ID": f"ID_{lid}",      # select: ID_1..ID_8
                "Behavior": "lying",  # preset
                "Hidden": "No"          # preset
            }

            annotations.append({
                "id": ann_id,
                "image_id": image_id,
                "category_id": 1,
                "bbox": [round(bbox[0],2), round(bbox[1],2), round(bbox[2],2), round(bbox[3],2)],
                "iscrowd": 0,
                "attributes": attr
            })
            ann_id += 1

coco = {"images": images, "annotations": annotations, "categories": categories}
with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(coco, f, ensure_ascii=False)
print(f"[OK] Wrote {OUT_JSON} | images={len(images)} | boxes={len(annotations)}")


Exporting COCO: 100%|██████████| 189/189 [07:06<00:00,  2.26s/it]

[OK] Wrote /content/drive/MyDrive/pig-selected_frame_attribute_(4)/images_frame_attribute_(4)/annotate_id.coco.json | images=1134 | boxes=8880
